## 1) Read the table using PySpark

In [0]:
# Read the table
df = spark.table("taxi")

In [0]:
# Display first 5 records
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |RateCodeID|store_and_fwd_flag|dropoff_longitude |dropoff_latitude  |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|1       |2015-01-28 20:21:35 |2015-01-28 20:35:06  |1              |2.4          |-74.00578308105469|40.750938415527344|1         |N   

In [0]:
# Check schema
df.printSchema()

root
 |-- VendorID: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- RateCodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



In [0]:
# Count total records
print("Total records:", df.count())

Total records: 12748986


## 2) Clean column names

In [0]:
import re
new_columns = [
    re.sub(
        r"_+",
        "_",
        re.sub(r"[^a-zA-Z0-9]+", "_", col.strip()).lower()
    ).strip("_")
    for col in df.columns
]
df = df.toDF(*new_columns)
df.printSchema()

root
 |-- vendorid: long (nullable = true)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- pickup_longitude: double (nullable = true)
 |-- pickup_latitude: double (nullable = true)
 |-- ratecodeid: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- dropoff_longitude: double (nullable = true)
 |-- dropoff_latitude: double (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)



## 3) Remove duplicate records

In [0]:
# Count records before removing duplicates
before_count = df.count()
print("Records before:", before_count)

Records before: 12748986


In [0]:
# Remove duplicates
df = df.dropDuplicates()

In [0]:
# Count records after removing duplicates
after_count = df.count()
print("Records after:", after_count)
print("Duplicates removed:", before_count - after_count)

Records after: 12748603
Duplicates removed: 383


## 4) Handle NULL values

In [0]:
# Check NULL values
from pyspark.sql.functions import col, sum, when
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|ratecodeid|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       0|                   0|                    0|              0|            0|               0|              0|         0|                 0|              

In [0]:
# Drop records missing VendorID
df = df.dropna(subset=["vendorid"])

In [0]:
# Fill remaining NULL values
from pyspark.sql.types import (
    StringType, IntegerType, LongType,
    ShortType, FloatType, DoubleType, DecimalType
)
numeric_types = (
    IntegerType, LongType, ShortType,
    FloatType, DoubleType, DecimalType
)
fill_values = {}
for field in df.schema.fields:
    if isinstance(field.dataType, numeric_types):
        fill_values[field.name] = 0
    elif isinstance(field.dataType, StringType):
        fill_values[field.name] = "unknown"
df = df.fillna(fill_values)

In [0]:
# Verify NULL values
df.select([
    sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]).show()

+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude|pickup_latitude|ratecodeid|store_and_fwd_flag|dropoff_longitude|dropoff_latitude|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+----------------+---------------+----------+------------------+-----------------+----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|       0|                   0|                    0|              0|            0|               0|              0|         0|                 0|              

## 5) Fix data types

In [0]:
# Convert data types
from pyspark.sql.types import (
    IntegerType, LongType, ShortType,
    FloatType, DoubleType, DecimalType
)
id_columns = [
    "vendorid",
    "ratecodeid",
    "pulocationid",
    "dolocationid",
    "payment_type"
]
numeric_types = (
    IntegerType, LongType, ShortType,
    FloatType, DoubleType, DecimalType
)
for field in df.schema.fields:
    if isinstance(field.dataType, numeric_types):
        if field.name not in id_columns:
            df = df.withColumn(
                field.name,
                col(field.name).cast("double")
            )
        else:
            df = df.withColumn(
                field.name,
                col(field.name).cast("int")
            )

In [0]:
# Verify data types
df.printSchema()

root
 |-- vendorid: integer (nullable = false)
 |-- tpep_pickup_datetime: timestamp (nullable = true)
 |-- tpep_dropoff_datetime: timestamp (nullable = true)
 |-- passenger_count: double (nullable = false)
 |-- trip_distance: double (nullable = false)
 |-- pickup_longitude: double (nullable = false)
 |-- pickup_latitude: double (nullable = false)
 |-- ratecodeid: integer (nullable = false)
 |-- store_and_fwd_flag: string (nullable = false)
 |-- dropoff_longitude: double (nullable = false)
 |-- dropoff_latitude: double (nullable = false)
 |-- payment_type: integer (nullable = false)
 |-- fare_amount: double (nullable = false)
 |-- extra: double (nullable = false)
 |-- mta_tax: double (nullable = false)
 |-- tip_amount: double (nullable = false)
 |-- tolls_amount: double (nullable = false)
 |-- improvement_surcharge: double (nullable = false)
 |-- total_amount: double (nullable = false)



In [0]:
# Display records
df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |ratecodeid|store_and_fwd_flag|dropoff_longitude |dropoff_latitude |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|2       |2015-01-29 20:14:08 |2015-01-29 20:44:00  |6.0            |11.98        |-73.96590423583984|40.764671325683594|1         |N      

## 6) Check negative amounts

In [0]:
# Identify monetary columns
amount_columns = [
    c for c in df.columns
    if "amount" in c or c in ["extra", "mta_tax"]
]
print("Monetary columns:", amount_columns)

Monetary columns: ['fare_amount', 'extra', 'mta_tax', 'tip_amount', 'tolls_amount', 'total_amount']


In [0]:
# Count negative values
from pyspark.sql.functions import col, sum, when
negative_counts = df.select([
    sum(
        when(col(c) < 0, 1).otherwise(0)
    ).alias(c)
    for c in amount_columns
])
negative_counts.show()

+-----------+-----+-------+----------+------------+------------+
|fare_amount|extra|mta_tax|tip_amount|tolls_amount|total_amount|
+-----------+-----+-------+----------+------------+------------+
|       4064| 1949|   3907|        79|          28|        4067|
+-----------+-----+-------+----------+------------+------------+



In [0]:
# Display records with negative amounts
from functools import reduce
from operator import or_
negative_condition = reduce(
    or_,
    [col(c) < 0 for c in amount_columns]
)
df.filter(negative_condition).show(20, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |ratecodeid|store_and_fwd_flag|dropoff_longitude |dropoff_latitude  |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+------------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|2       |2015-01-12 14:44:49 |2015-01-12 14:44:54  |1.0            |0.0          |0.0               |0.0               |2         |N   

In [0]:
# Stop execution if negative amounts exist
negative_records = df.filter(
    negative_condition
).limit(1).count()
if negative_records > 0:
    raise ValueError("Negative amounts found! Ingestion stopped.")
else:
    print("Validation passed! No negative amounts found.")

---------------------------------------------------------------------------
ValueError                                Traceback (most recent call last)
File <command-6183259633523873>, line 6
      2 negative_records = df.filter(
      3     negative_condition
      4 ).limit(1).count()
      5 if negative_records > 0:
----> 6     raise ValueError("Negative amounts found! Ingestion stopped.")
      7 else:
      8     print("Validation passed! No negative amounts found.")

ValueError: Negative amounts found! Ingestion stopped.

## 7) Save as Parquet

In [0]:
# Create the Volume
spark.sql("""
CREATE VOLUME IF NOT EXISTS
workspace.default.nyc_taxi_volume
""")
print("Volume created successfully!")

Volume created successfully!


In [0]:
# Show volume
spark.sql("""
SHOW VOLUMES IN workspace.default
""").show(truncate=False)

+--------+---------------+
|database|volume_name    |
+--------+---------------+
|default |nyc_taxi_volume|
+--------+---------------+



In [0]:
# Define the output path
output_path = "/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet"

In [0]:
# Save the cleaned DataFrame as Parquet
df.write \
    .format("parquet") \
    .mode("overwrite") \
    .save(output_path)
print("NYC Taxi Parquet file saved successfully!")

NYC Taxi Parquet file saved successfully!


In [0]:
# Read the saved Parquet file
clean_df = spark.read \
    .format("parquet") \
    .load(output_path)
clean_df.show(5, truncate=False)

+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|vendorid|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|pickup_longitude  |pickup_latitude   |ratecodeid|store_and_fwd_flag|dropoff_longitude |dropoff_latitude |payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|
+--------+--------------------+---------------------+---------------+-------------+------------------+------------------+----------+------------------+------------------+-----------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+
|1       |2015-01-10 18:26:17 |2015-01-10 18:32:24  |1.0            |0.8          |-73.99250030517578|40.745399475097656|1         |N      

In [0]:
# Count the cleaned records
print("Total cleaned records:", clean_df.count())

Total cleaned records: 12748603


In [0]:
# Verify the saved files
display(dbutils.fs.ls(output_path))

path,name,size,modificationTime
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/_SUCCESS,_SUCCESS,0,1795053217000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/_committed_2544668417472075374,_committed_2544668417472075374,2234,1795053217000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/_committed_5185316818189346648,_committed_5185316818189346648,1124,1795053064000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/_started_2544668417472075374,_started_2544668417472075374,0,1795053214000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/_started_5185316818189346648,_started_5185316818189346648,0,1795053061000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/part-00000-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-399-1.c000.snappy.parquet,part-00000-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-399-1.c000.snappy.parquet,38005432,1795053215000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/part-00001-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-403-1.c000.snappy.parquet,part-00001-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-403-1.c000.snappy.parquet,37936842,1795053215000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/part-00002-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-404-1.c000.snappy.parquet,part-00002-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-404-1.c000.snappy.parquet,38132969,1795053215000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/part-00003-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-405-1.c000.snappy.parquet,part-00003-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-405-1.c000.snappy.parquet,38020238,1795053215000
dbfs:/Volumes/workspace/default/nyc_taxi_volume/nyc_taxi_clean.parquet/part-00004-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-400-1.c000.snappy.parquet,part-00004-tid-2544668417472075374-240f63eb-d47d-4f3f-a5bb-73c9824f8708-400-1.c000.snappy.parquet,38074325,1795053215000


## 8) Logging and error handling

In [0]:
# Import logging
import logging
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s - %(levelname)s - %(message)s"
)
logger = logging.getLogger("NYC_Taxi_Ingestion")

In [0]:
# Log each ingestion step
logger.info("Step 1: Reading NYC Taxi table completed")
logger.info("Step 2: Column names cleaned")
logger.info("Step 3: Duplicate records removed")
logger.info("Step 4: NULL values handled")
logger.info("Step 5: Data types converted")
logger.info("Step 6: Negative amount validation passed")
logger.info("Step 7: Data saved as Parquet")
logger.info("Step 8: Ingestion completed successfully")

2026-11-19 01:55:31,811 - INFO - Step 1: Reading NYC Taxi table completed
2026-11-19 01:55:31,812 - INFO - Step 2: Column names cleaned
2026-11-19 01:55:31,813 - INFO - Step 3: Duplicate records removed
2026-11-19 01:55:31,813 - INFO - Step 4: NULL values handled
2026-11-19 01:55:31,814 - INFO - Step 5: Data types converted
2026-11-19 01:55:31,815 - INFO - Step 6: Negative amount validation passed
2026-11-19 01:55:31,816 - INFO - Step 7: Data saved as Parquet
2026-11-19 01:55:31,817 - INFO - Step 8: Ingestion completed successfully


In [0]:
# Implement try-except error handling
try:
    logger.info("Starting NYC Taxi ingestion")
    df = spark.table("taxi")
    logger.info("Source table loaded successfully")
    df.write \
        .format("parquet") \
        .mode("overwrite") \
        .save(output_path)
    logger.info("Parquet file saved successfully")
    clean_df = spark.read.parquet(output_path)
    logger.info(
        "Total cleaned records: %d",
        clean_df.count()
    )
    logger.info("NYC Taxi ingestion completed successfully")
except Exception as e:
    logger.exception("NYC Taxi ingestion failed")
    raise

2026-11-19 01:56:15,489 - INFO - Starting NYC Taxi ingestion
2026-11-19 01:56:15,490 - INFO - Source table loaded successfully
2026-11-19 01:56:19,123 - INFO - Parquet file saved successfully
2026-11-19 01:56:20,150 - INFO - Total cleaned records: 12748986
2026-11-19 01:56:20,150 - INFO - NYC Taxi ingestion completed successfully
